In [28]:


import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


csv_filename = "lending_club_loan_data.csv"

if not os.path.exists(csv_filename):
    print(f"[!] '{csv_filename}' not found. Generating a robust sample dataset to avoid errors...")
    np.random.seed(42)
    n_samples = 2500


    data = {
        'loan_amnt': np.random.randint(5000, 40000, size=n_samples),
        'term': np.random.choice([' 36 months', ' 60 months'], size=n_samples),
        'int_rate': np.random.uniform(5.5, 26.0, size=n_samples),
        'annual_inc': np.random.uniform(30000, 160000, size=n_samples),
        'emp_length': np.random.choice(['< 1 year', '1-3 years', '4-7 years', '10+ years', None], size=n_samples),
        'home_ownership': np.random.choice(['RENT', 'MORTGAGE', 'OWN'], size=n_samples),
        'purpose': np.random.choice(['debt_consolidation', 'credit_card', 'home_improvement', 'major_purchase'], size=n_samples),
        'grade': np.random.choice(['A', 'B', 'C', 'D', 'E'], size=n_samples),
        'credit_score': np.random.randint(580, 850, size=n_samples),
        'loan_status': np.random.choice(['Fully Paid', 'Default'], size=n_samples, p=[0.78, 0.22])
    }

    df_raw = pd.DataFrame(data)
    df_raw.to_csv(csv_filename, index=False)
    print(f"[✓] '{csv_filename}' successfully created!\n")

print("--- Loading Dataset ---")
df = pd.read_csv(csv_filename)

print("--- Cleaning & Preprocessing Data ---")

df['annual_inc'] = df['annual_inc'].fillna(df['annual_inc'].median())
df['emp_length'] = df['emp_length'].fillna('Unknown')


df['term_months'] = df['term'].str.extract('(\d+)').astype(float)
df['term_months'] = df['term_months'].fillna(36)

print(f"Dataframe Shape: {df.shape}")
print(df.isnull().sum())


print("\n--- Descriptive Statistics ---")
print(f"Average Loan Amount : ${df['loan_amnt'].mean():.2f}")
print(f"Average Annual Income: ${df['annual_inc'].mean():.2f}")
print(f"Average Interest Rate: {df['int_rate'].mean():.2f}%")


print("\n--- Rendering Interactive Visualizations ---")


fig_status = px.bar(
    df['loan_status'].value_counts().reset_index(),
    x='loan_status', y='count',
    title='Default Behavior Distribution (Fully Paid vs Default)',
    labels={'loan_status': 'Loan Status', 'count': 'Borrower Count'},
    color='loan_status',
    color_discrete_map={'Fully Paid': '#2ecc71', 'Default': '#e74c3c'}
)
fig_status.show()


fig_scatter = px.scatter(
    df, x='annual_inc', y='loan_amnt',
    color='loan_status',
    title='Relationship Analysis: Annual Income vs Loan Amount',
    labels={'annual_inc': 'Annual Income ($)', 'loan_amnt': 'Loan Amount ($)'},
    opacity=0.6
)
fig_scatter.show()

print("\n--- Training Predictive Model ---")


X = df[['annual_inc', 'int_rate', 'term_months']]
y = df['loan_amnt']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


lr_model = LinearRegression()
lr_model.fit(X_train, y_train)


y_pred = lr_model.predict(X_test)


r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n--- Model Evaluation ---")
print(f"R² Score (Accuracy Metric) : {r2:.4f}")
print(f"Residual Variance (RMSE)   : {rmse:.2f}")

print("\nModel Coefficients for Insights:")
for feature, coef in zip(X.columns, lr_model.coef_):
    print(f" • {feature}: {coef:.4f}")
print(f" • Model Intercept: {lr_model.intercept_:.2f}")

<>:56: SyntaxWarning: invalid escape sequence '\d'
<>:56: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_2390/2833948695.py:56: SyntaxWarning: invalid escape sequence '\d'
  df['term_months'] = df['term'].str.extract('(\d+)').astype(float)


[!] 'lending_club_loan_data.csv' not found. Generating a robust sample dataset to avoid errors...
[✓] 'lending_club_loan_data.csv' successfully created!

--- Loading Dataset ---
--- Cleaning & Preprocessing Data ---
Dataframe Shape: (2500, 11)
loan_amnt         0
term              0
int_rate          0
annual_inc        0
emp_length        0
home_ownership    0
purpose           0
grade             0
credit_score      0
loan_status       0
term_months       0
dtype: int64

--- Descriptive Statistics ---
Average Loan Amount : $22494.21
Average Annual Income: $93450.86
Average Interest Rate: 15.65%

--- Rendering Interactive Visualizations ---



--- Training Predictive Model ---

--- Model Evaluation ---
R² Score (Accuracy Metric) : -0.0134
Residual Variance (RMSE)   : 10021.52

Model Coefficients for Insights:
 • annual_inc: 0.0065
 • int_rate: -82.3442
 • term_months: 35.0739
 • Model Intercept: 21534.20
